# Apartado 4. Análisis de subjetividad de comentarios

En este apartado, vamos a emplear un modelo preentrenado disponible en HuggingFace, concretamente https://huggingface.co/cardiffnlp/twitter-roberta-base-sentiment, para realizar un análisis de sentimientos sobre los comentarios de nuestros subreddits. Este modelo clasifica cada comentario en una de las tres categorías: Positivo, negativo, neutro.

Por simplicidad haremos uso de los datos preprocesados

In [1]:
!pip install transformers

In [2]:
from transformers import AutoModelForSequenceClassification
from transformers import AutoTokenizer
import numpy as np
from scipy.special import softmax
import csv
import urllib.request
import json
import torch

In [4]:
subreddits = ["limpio_books.json", "limpio_jobs.json", "limpio_LeagueOfLegends.json", "limpio_RandomThoughts.json",
			  "limpio_travel.json", "limpio_unpopularopinion.json"]

MAX_EJEMPLOS = 5

task='sentiment'
MODEL = f"cardiffnlp/twitter-roberta-base-{task}"

tokenizer = AutoTokenizer.from_pretrained(MODEL)

# download label mapping
labels=[]
mapping_link = f"https://raw.githubusercontent.com/cardiffnlp/tweeteval/main/datasets/{task}/mapping.txt"
with urllib.request.urlopen(mapping_link) as f:
    html = f.read().decode('utf-8').split("\n")
    csvreader = csv.reader(html, delimiter='\t')
labels = [row[1] for row in csvreader if len(row) > 1]

# PT
model = AutoModelForSequenceClassification.from_pretrained(MODEL)

for subreddit in subreddits:
	with open(subreddit, "r", encoding="utf-8") as file:
		datos = json.load(file)

		sentimiento = {}
		ejemplos_mostrados = 0

		# Extraemos todos los hilos pertenecientes al subreddit
		hilos = datos["submissions"]
		for hilo in hilos:
			for comentario in hilo["comments"]:
				# Como el modelo tiene un límite de 512 tokens, truncamos los textos que lo superan
				encoded_input = tokenizer(comentario["body"].lower(), return_tensors='pt', truncation=True, max_length=512)

				# Añadimos esto para que no se guarden gradientese innecesarios ya que no estamos entrenando el modelo, solo prediciendo el sentimiento
				with torch.no_grad():
					output = model(**encoded_input)
					scores = output[0][0].detach().numpy()
					scores = softmax(scores)

					ranking = np.argsort(scores)[::-1]

					# Añadimos el campo sentiment como se pedía, y además los scores de los demás sentimientos en scores
					comentario["sentiment"] = labels[ranking[0]]
					comentario["scores"] = {}

					for i in range(scores.shape[0]):
						l = labels[ranking[i]]
						s = scores[ranking[i]]
						comentario["scores"][l] = round(float(s), 4)

					if ejemplos_mostrados < MAX_EJEMPLOS:
						print(f"\nComentario: {comentario["body"]}")
						print(f"Predicción: {comentario["sentiment"]}")
						print(f"Scores: {comentario["scores"]}")
						ejemplos_mostrados += 1

		# Guardamos el json modificado
		nombre_salida = f"sentimientos_{subreddit}"

		with open(nombre_salida, "w", encoding="utf-8") as f:
			json.dump(datos, f, ensure_ascii=False, indent=4)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Comentario: You 're preaching choir
Predicción: neutral
Scores: {'neutral': 0.699, 'negative': 0.2283, 'positive': 0.0727}

Comentario: Does listening radio count reading Especially song like Devil Went Down Georgia tells story Does listening somebody talk count reading It silly conversation counts someone wants ask questions
Predicción: neutral
Scores: {'neutral': 0.8033, 'negative': 0.1411, 'positive': 0.0556}

Comentario: refer audiobooks bookbooks talking friends It 's stupid 's stuck also give anyone says `` audiobooks n't real books/reading '' quick boot arse say r/Discworld sub We class form ableism tbh Consume media choose consume Watch films subtitles listen books play computer games console choice As long 're happy 's important thing
Predicción: negative
Scores: {'negative': 0.5493, 'neutral': 0.3757, 'positive': 0.075}

Comentario: Are many people saying otherwise
Predicción: neutral
Scores: {'neutral': 0.6223, 'negative': 0.3531, 'positive': 0.0246}

Comentario: call A/V r